# 🌱 DriveSeed: Google Colab Edition

Welcome to **DriveSeed Colab Edition**!
This notebook runs the DriveSeed download and library manager inside your Google Colab runtime, storing files directly in your Google Drive.

### 🛡️ Colab Compliance Guarantee
- **Torrents Disabled**: Torrent downloading is completely disabled in accordance with Google Colab Terms of Service.
- **Direct Downloads**: High-speed multi-threaded direct HTTP/HTTPS downloads directly landed into Google Drive.
- **Zero Media Proxying**: Media streaming is handed off directly to Google Drive's native viewer.
- **Persistent State**: SQLite database is stored on your Google Drive (`/content/drive/MyDrive/DriveSeed/.driveseed.db`) to survive runtime resets.

## Step 1: Mount Google Drive
Mount your Google Drive to `/content/drive`. Your downloads and database will be saved in `MyDrive/DriveSeed`.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/DriveSeed')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ Google Drive mounted. Target directory ready: {DRIVE_ROOT}")

## Step 2: Install Dependencies
Install the lightweight Colab-specific requirements (omitting heavy torrent/redis binaries).

In [ ]:
# If running directly from git repository or local upload:
!pip install -q fastapi uvicorn pydantic pydantic-settings httpx sqlalchemy aiosqlite greenlet cryptography alembic
print("✓ Dependencies installed.")

## Step 3: Configure Environment Variables
Set runtime profile to `colab` and configure persistent SQLite storage on Google Drive.

In [ ]:
import os
import secrets
import base64

os.environ["PROFILE"] = "colab"
os.environ["COLAB_DRIVE_ROOT"] = "/content/drive/MyDrive/DriveSeed"
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:////content/drive/MyDrive/DriveSeed/.driveseed.db"
os.environ["SECRET_KEY"] = os.environ.get("SECRET_KEY", secrets.token_hex(32))
if "ENCRYPTION_KEY" not in os.environ:
    os.environ["ENCRYPTION_KEY"] = base64.b64encode(secrets.token_bytes(32)).decode()

print("✓ Environment configured:")
print(f"  PROFILE = {os.environ['PROFILE']}")
print(f"  COLAB_DRIVE_ROOT = {os.environ['COLAB_DRIVE_ROOT']}")
print(f"  DATABASE_URL = {os.environ['DATABASE_URL']}")

## Step 4: Initialize Database & Start DriveSeed Server
Run migrations or initialize tables, then launch the FastAPI server in the background.

In [ ]:
import sys
import subprocess
import time

# Ensure backend directory is in Python path
backend_dir = os.path.abspath('backend')
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

# Initialize SQLite schema
import asyncio
from app.core.db import get_engine, Base

async def init_db():
    engine = get_engine()
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    print("✓ Database tables initialized on Google Drive.")

loop = asyncio.get_event_loop()
if loop.is_running():
    import nest_asyncio
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        !pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()
loop.run_until_complete(init_db())

# Launch uvicorn server in background
server_process = subprocess.Popen(
    ["uvicorn", "app.main:create_app", "--factory", "--host", "127.0.0.1", "--port", "8000"],
    cwd=backend_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print("✓ DriveSeed server running on port 8000.")

## Step 5: Display DriveSeed UI (Option B)
Serve the interactive React web application directly inside Google Colab.

In [ ]:
from google.colab import output

# Render the web application iframe inside the Colab cell output
output.serve_kernel_port_as_iframe(8000, height=800)